In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import logging
import warnings
warnings.filterwarnings('ignore')

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

# BaoStockDB

In [3]:
# 创建因子库对象并 connect
from QuantStudio.Factor.BaoStockDB import BaoStockDB

FDB = BaoStockDB().connect()
print(qs_help(FDB))

2026-05-13 14:59:55,197 | QS | WARNING : 找不到配置文件: C:\Users\lenovo\QuantStudioConfig\BaoStockDBConfig.json


login success!
类型: BaoStockDB
模块: QuantStudio.Factor.BaoStockDB
QS 对象类型: 因子库
QS 对象名称: BaoStockDB
QSID: 3fdb5be21a9b27b10903497701dbea3e8d35a0a3a851b23c2e4219e6fd91a7e0
参数集:
    * Name(名称): <class 'str'>, 默认值 'BaoStockDB', 当前取值: 'BaoStockDB'
说明文档:
    基于 BaoStock 的因子库
    API: http://baostock.com/baostock/
    库配置信息文件在 QuantStudio 包目录下 Resource 目录下的 BaoStockDBInfo.xlsx, 记录了相关配置信息


In [5]:
# 获取因子库中的因子表列表
print(FDB.TableNames[:5])

['A股K线数据', '行业分类']


## 获取时点序列

### 获取交易日序列

In [6]:
# 获取交易日序列的方法说明
print(qs_help(FDB.getTradeDay))

类型: method (bound to BaoStockDB)
模块: QuantStudio.Factor.BaoStockDB
签名: BaoStockDB.getTradeDay(start_date: Optional[datetime.datetime] = None, end_date: Optional[datetime.datetime] = None, exchange: Literal['SSE'] = 'SSE', **kwargs) -> List[datetime.datetime]
说明文档:
    给定交易所、起始日和结束日, 获取交易日序列
    
    Args:
        start_date: 起始日, None 表示从可取的最早日期开始
        end_date: 结束日, None 表示当前日期
        exchange: 交易所, 默认 SSE(上交所)
    
    Returns:
        交易日序列


In [7]:
# 给定起止时点, 获取交易日序列
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 20))
print(DTs[:5])

login success!
[Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-05 00:00:00'), Timestamp('2022-01-06 00:00:00'), Timestamp('2022-01-07 00:00:00'), Timestamp('2022-01-10 00:00:00')]


## 获取证券代码序列

### 股票证券代码

In [8]:
# 获取股票证券代码方法说明
print(qs_help(FDB.getStockID))

类型: method (bound to BaoStockDB)
模块: QuantStudio.Factor.BaoStockDB
签名: BaoStockDB.getStockID(exchange: Union[str, Tuple[str], NoneType] = ('SSE', 'SZSE'), date: Optional[datetime.datetime] = None, is_current: bool = True, **kwargs) -> List[str]
说明文档:
    给定交易所和日期, 获取股票证券 ID 序列
    
    Args:
        exchange: 交易所(str)或者交易所列表(tuple), 默认 ("SSE", "SZSE") 表示上交所、深交所
        date: 指定日, 默认值 None 表示当前日期
        is_current: False 表示上市日期在指定日之前的股票, True 表示上市日期在指定日之前且尚未退市的股票
        
    Returns:
        股票证券 ID 序列


In [10]:
# 获取全体A股，不包括已经退市的
IDs = FDB.getStockID()
print(IDs[:5])

login success!
['000001.SZ', '000002.SZ', '000006.SZ', '000007.SZ', '000008.SZ']


# 因子表

In [4]:
# 获取因子表对象
FT = FDB.getTable("A股K线数据", args={"LookBack": 0})
print(qs_help(FT))

类型: _DTRangeTable
模块: QuantStudio.Factor.BaoStockDB
QS 对象类型: 计算节点-因子表
QS 对象名称: A股K线数据
QSID: 1bcfc1b16ab117c15d435df31ea8ba3be60a82ed80479419e2727e79ab857eb3
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'A股K线数据'
    * LookBack(回溯天数): <class 'int'>, 默认值 0, 当前取值: 0
说明文档:
    BaoStockDB 库中基于取时间区间数据 API 的因子表


In [6]:
# 因子列表
print(FT.FactorNames)

['open', 'high', 'low', 'close', 'preclose', 'volume', 'amount', 'adjustflag', 'turn', 'tradestatus', 'pctChg', 'peTTM', 'pbMRQ', 'psTTM', 'pcfNcfTTM', 'isST']


## 读取数据

In [7]:
# 因子表读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据")
print(Data)

因子表数据
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: open to close
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


# 因子

In [8]:
# 获取因子对象
F = FT.getFactor("close")
print(qs_help(F))

类型: Factor
模块: QuantStudio.Factor.Factor
QS 对象类型: 计算节点-因子
QS 对象名称: close
QSID: 67e7721908f04d24974f5163c5b0be3a671a62aecb25d0b2162af0d9ac22d68f
参数集:
    * Name(名称): <class 'str'>, 默认值 'Factor', 当前取值: 'close'
    * Meta(元信息): <class 'dict'>, 默认值 {}, 当前取值: {}
    * SectionIDs(截面ID): typing.Optional[typing.List[str]], 默认值 None, 当前取值: None
    * CalcDTRuler(计算时点标尺): typing.Optional[typing.List[datetime.datetime]], 默认值 None, 当前取值: None
说明文档:
    因子对象
    因子可看做 DataFrame(index=[时点], columns=[ID])
    时点数据类型是 datetime, ID 的数据类型是 str


## 读取数据

In [9]:
# 因子读取数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = F.readData(ids=IDs, dts=DTs)
print(Data)

              000001.SZ   000002.SZ
2025-01-01          NaN         NaN
2025-01-02  1312.459523  966.153378
2025-01-03  1306.718230  951.205857
2025-01-04          NaN         NaN
2025-01-05          NaN         NaN
